# 01 · Data
SPX monthly option chains (ThetaData), SPX OHLC (Stooq), VIX and Treasury yields (FRED), SKEW (yfinance).
Raw files are produced by `scripts/download_data.py` and are **not** redistributed. This notebook only shows aggregates.

In [1]:
from pathlib import Path
import pandas as pd
from vrp import data, features, models, backtest, plots

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
pd.set_option("display.precision", 4)

In [2]:
raw = data.load_raw(RAW)
{k: (len(v), v["date"].min().date(), v["date"].max().date()) for k, v in raw.items()}

{'underlying': (2011, datetime.date(2018, 1, 2), datetime.date(2025, 12, 31)),
 'macro': (2040, datetime.date(2018, 1, 2), datetime.date(2025, 12, 31)),
 'chain': (341094, datetime.date(2018, 1, 2), datetime.date(2025, 12, 4))}

Cleaning filters applied during download (thesis §3.2):

In [3]:
cfg = data.DataConfig()
{k: getattr(cfg, k) for k in ["min_open_interest", "max_rel_spread", "dte_min", "dte_max", "moneyness_min", "moneyness_max"]}

{'min_open_interest': 100,
 'max_rel_spread': 0.2,
 'dte_min': 15,
 'dte_max': 45,
 'moneyness_min': 0.85,
 'moneyness_max': 1.15}

Coverage of the cleaned chain by year. **2021 is empty**: the chains for that year could not be retrieved in the thesis run, so 2021 has no option features and the backtest does not trade that year.

In [4]:
chain = raw["chain"]
(chain.assign(year=chain["date"].dt.year)
      .groupby("year").agg(rows=("strike", "size"), sessions=("date", "nunique"), expirations=("expiration", "nunique"))
      .reindex(range(2018, 2026), fill_value=0))

,rows,sessions,expirations
year,,,
2018,38063,243,13
2019,35549,224,12
2020,33206,196,11
2021,0,0,0
2022,61314,225,12
2023,54748,243,13
2024,60031,240,13
2025,58183,205,11


In [5]:
raw["macro"].describe().T[["count", "mean", "min", "max"]]

,count,mean,min,max
date,2040,2022-01-09 21:21:10.588235,2018-01-02 00:00:00,2025-12-31 00:00:00
macro_VIX,2038.0,19.7312,9.15,82.69
macro_UST_3M,1999.0,2.6518,0.0,5.63
macro_UST_10Y,1999.0,2.8474,0.52,4.98
macro_SKEW,1960.0,136.1352,110.34,183.12
